# RAR — Real Context-Rot (CRS) Trajectory
Run cells **top to bottom**. Requires an `OPENROUTER_API_KEY` Colab Secret (key icon, notebook access ON). Uses the **instrumented** repo, so per-cycle data is logged and the CRS trajectory is genuine — no simulation.


### 1. Clone instrumented repo + install

In [ ]:
import os
REPO_DIR = "/content/recursive-autonomy-research"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tahir-yamin/recursive-autonomy-research.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull          # pulls the INSTRUMENTED main
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
# confirm the per-cycle instrumentation is present in this clone
assert "per_cycle_trajectories" in open("run_pilot_experiment.py").read(), \
    "run_pilot_experiment.py is NOT instrumented -- pull main again."
print("Instrumentation present: OK")
!pip install -q torch numpy scipy scikit-learn networkx aiohttp matplotlib

### 2. Load API key (real-inference mode)

In [ ]:
import os
# OPENROUTER_API_KEY must be added as a Colab Secret (key icon) with notebook access ON.
try:
    from google.colab import userdata
    key = userdata.get("OPENROUTER_API_KEY")
except Exception as e:
    key = None; print("Could not read Colab Secret:", e)
if not key:
    raise ValueError("No key found. Add OPENROUTER_API_KEY as a Colab Secret, enable "
                     "notebook access, and re-run. (Do NOT set RAR_SIM -- that is the fake path.)")
os.environ["OPENROUTER_API_KEY"] = key
os.environ.pop("RAR_SIM", None)        # ensure the real path
os.environ["RAR_CYCLES"] = "60"        # cycles per seed (60 = full horizon)
print(f"Key loaded (...{key[-6:]}). Real-inference mode.")

### 3. Pre-flight: auto-select a working free model

In [ ]:
import sys, os
os.chdir("/content/recursive-autonomy-research")
for m in ("run_pilot_experiment", "run_deep_learning_harness"):
    sys.modules.pop(m, None)
import run_pilot_experiment as rpe
# Free model slugs rotate; pick the first that answers.
CANDIDATES = [
    "openai/gpt-oss-20b:free",
    "meta-llama/llama-3.3-70b-instruct:free",
    "google/gemma-3-12b-it:free",
    "deepseek/deepseek-chat-v3:free",
    "qwen/qwen-2.5-72b-instruct:free",
]
chosen = None
for slug in CANDIDATES:
    os.environ["OPENROUTER_MODEL"] = slug
    try:
        resp = await rpe.call_llm("Reply with the single word: OK")
    except Exception as e:
        print(f"  x {slug:45s} {str(e)[:70]}"); continue
    if resp:
        chosen = slug; print(f"  ok {slug:45s} -> {resp[:30]!r}"); break
if not chosen:
    raise RuntimeError("PRE-FLIGHT FAILED: no free model responded. "
                       "See https://openrouter.ai/models?max_price=0")
os.environ["OPENROUTER_MODEL"] = chosen
print("\nPRE-FLIGHT OK -> model locked:", chosen)

### 4. Init campaign (dataset = digits)

In [ ]:
import os, sys
os.chdir("/content/recursive-autonomy-research")
DATASET = "digits"                     # real 10-class task
os.environ["RAR_DATASET"] = DATASET
os.environ["RAR_CYCLES"] = os.environ.get("RAR_CYCLES", "60")
for m in ("run_pilot_experiment", "run_deep_learning_harness"):
    sys.modules.pop(m, None)
import run_pilot_experiment as rpe      # re-import so RAR_DATASET takes effect

async def run_one_seed(seed):
    out = f"pilot_seed_{seed}_{DATASET}.json"
    if os.path.exists(out):
        print(f"Seed {seed}: already done, skipping."); return
    print(f"===== SEED {seed} [{DATASET}] =====")
    os.environ["RAR_OUTPUT_FILE"] = out
    rpe.SEEDS = [seed]; rpe.CYCLES = int(os.environ["RAR_CYCLES"])
    await rpe.execute_campaign()
    print(f"Seed {seed}: DONE -> {out}" if os.path.exists(out) else f"WARN: no file for {seed}")
print("Init OK for", DATASET)

### 5. Run the real campaign

In [ ]:
# Run the real campaign. 3 seeds is enough for a trajectory + CI band; add the
# rest (88, 99, 101, 107, 113, 127) for the full N=10. Each seed = 3 conditions x
# 60 cycles of real LLM calls (minutes each; free tier may rate-limit -- it retries).
SEEDS = [42, 7, 13]
for s in SEEDS:
    await run_one_seed(s)
print("Campaign complete for seeds:", SEEDS)

### 6. Compute & plot the REAL CRS trajectory

In [ ]:
# ============================================================================
# CORRECTED Cell 26 -- REAL Context-Rot (CRS) trajectory.
# Replaces the broken version that plotted per-SEED redundancy counts as if
# they were cycles. This reads the genuine per-cycle trajectories that the
# INSTRUMENTED run_pilot_experiment.py now logs under
# conditions[*]["per_cycle_trajectories"]. If that field is absent (i.e. the
# campaign was run with the OLD orchestrator), it refuses and tells you to
# re-run -- it does NOT fabricate a trajectory.
#
# Requires: your repo's run_pilot_experiment.py must be the instrumented
# version (it persists per_cycle_trajectories). Re-run the seed cells (13-22)
# AFTER updating the repo, then run this cell.
# ============================================================================
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt

W = 10  # trailing window
COLORS = {"stateless_baseline": "#b41e1e", "vector_rag": "#1c4f8a", "rar_compressed": "#2ca02c"}
LABELS = {"stateless_baseline": "Stateless", "vector_rag": "Vector RAG", "rar_compressed": "RAR (ours)"}

# Prefer per-seed files (each has one trajectory); fall back to merged file.
DATASET = os.environ.get("RAR_DATASET", "digits")
per_seed = sorted(glob.glob(f"pilot_seed_*_{DATASET}.json"))
conds = {}
if per_seed:
    for f in per_seed:
        c = json.load(open(f)).get("data", {}).get("conditions", {})
        for k, v in c.items():
            conds.setdefault(k, {"per_cycle_trajectories": []})
            conds[k]["per_cycle_trajectories"] += v.get("per_cycle_trajectories", [])
else:
    merged = f"pilot_results_{DATASET}.json"
    if os.path.exists(merged):
        conds = json.load(open(merged)).get("data", {}).get("conditions", {})

have = any(conds.get(c, {}).get("per_cycle_trajectories") for c in conds)
if not have:
    raise SystemExit(
        "No per_cycle_trajectories found. Your campaign was run with the OLD "
        "orchestrator (per-seed means only). Update run_pilot_experiment.py to "
        "the instrumented version, re-run the seed cells, then run this cell. "
        "No trajectory will be fabricated.")


def crs_series(trace):
    trace = sorted(trace, key=lambda r: r["cycle"])
    red = np.array([r["redundant"] for r in trace], float)
    acc = np.array([r["val_acc"] for r in trace], float)
    n = len(trace); NR = np.empty(n); DC = np.empty(n)
    for t in range(n):
        lo = max(0, t - W + 1)
        NR[t] = 1.0 - red[lo:t + 1].mean()
        win = acc[lo:t + 1]
        prior = np.array([acc[:max(1, lo + i)].max() for i in range(len(win))])
        DC[t] = np.mean(win >= prior)
    return (NR + DC) / 2.0, np.array([r["density"] for r in trace])


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
for c in ["stateless_baseline", "vector_rag", "rar_compressed"]:
    trs = [t for t in conds.get(c, {}).get("per_cycle_trajectories", []) if t and len(t) > 1]
    if not trs:
        continue
    L = min(len(t) for t in trs)
    crs = np.array([crs_series(t)[0][:L] for t in trs])
    den = np.array([crs_series(t)[1][:L] for t in trs])
    x = np.arange(1, L + 1)
    ax1.plot(x, crs.mean(0), color=COLORS[c], lw=2, label=LABELS[c])
    ax1.fill_between(x, crs.mean(0) - crs.std(0), crs.mean(0) + crs.std(0), color=COLORS[c], alpha=0.15)
    ax2.plot(x, den.mean(0), color=COLORS[c], lw=2, label=LABELS[c])
ax1.axhline(0.70, color="black", ls=":", lw=1, label="success (0.70)")
ax1.set_xlabel("Cycle"); ax1.set_ylabel("CRS"); ax1.set_title("Coherence Retention Score trajectory"); ax1.legend(fontsize=8); ax1.grid(alpha=0.4)
ax2.set_xlabel("Cycle"); ax2.set_ylabel(r"Context density $\delta$"); ax2.set_title("Context density trajectory"); ax2.legend(fontsize=8); ax2.grid(alpha=0.4)
plt.tight_layout(); plt.savefig("fig_rot_trajectory.png", dpi=300, bbox_inches="tight"); plt.show()
print("Saved REAL fig_rot_trajectory.png  (download this + pilot_results_digits.json and send back).")


### 7. Show figure + download outputs

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("fig_rot_trajectory.png"):
    display(Image("fig_rot_trajectory.png"))
# bundle the files to send back for manuscript embedding
import glob, zipfile
with zipfile.ZipFile("rar_crs_outputs.zip", "w") as z:
    for f in glob.glob("pilot_seed_*_digits.json") + ["fig_rot_trajectory.png"]:
        if os.path.exists(f): z.write(f)
try:
    from google.colab import files; files.download("rar_crs_outputs.zip")
except Exception as e:
    print("Download manually: rar_crs_outputs.zip", e)
print("Send rar_crs_outputs.zip back for manuscript embedding.")